# Graph Convolutional Network (late simulations)

In [55]:
batch_size = 8 
data_files = [
    {"label": [0], "topology_filename": "../FP11_monomer/SYSTEMns.prmtop", "coordinates_filename": "../FP11_monomer/PRODUCTION_0/TRAJ-last75ns.nc"},
    {"label": [0], "topology_filename": "../FP11_monomer/SYSTEMns.prmtop", "coordinates_filename": "../FP11_monomer/PRODUCTION_1/TRAJ-last75ns.nc"},
    {"label": [0], "topology_filename": "../FP11_monomer/SYSTEMns.prmtop", "coordinates_filename": "../FP11_monomer/PRODUCTION_2/TRAJ-last75ns.nc"},
    {"label": [1], "topology_filename": "../FP12-TLR4-MDago_monomer/SYSTEMns.prmtop", "coordinates_filename": "../FP12-TLR4-MDago_monomer/PRODUCTION_0/TRAJ-last75ns.nc"},
    {"label": [1], "topology_filename": "../FP12-TLR4-MDago_monomer/SYSTEMns.prmtop", "coordinates_filename": "../FP12-TLR4-MDago_monomer/PRODUCTION_1/TRAJ-last75ns.nc"},
    {"label": [1], "topology_filename": "../FP12-TLR4-MDago_monomer/SYSTEMns.prmtop", "coordinates_filename": "../FP12-TLR4-MDago_monomer/PRODUCTION_2/TRAJ-last75ns.nc"}
]
selection_string = "name CA N C"
epochs = 15
test_size = 0.20
random_state = 42
lr = 0.0001

In [56]:
data_files

[{'label': [0],
  'topology_filename': '../FP11_monomer/SYSTEMns.prmtop',
  'coordinates_filename': '../FP11_monomer/PRODUCTION_0/TRAJ-last75ns.nc'},
 {'label': [0],
  'topology_filename': '../FP11_monomer/SYSTEMns.prmtop',
  'coordinates_filename': '../FP11_monomer/PRODUCTION_1/TRAJ-last75ns.nc'},
 {'label': [0],
  'topology_filename': '../FP11_monomer/SYSTEMns.prmtop',
  'coordinates_filename': '../FP11_monomer/PRODUCTION_2/TRAJ-last75ns.nc'},
 {'label': [1],
  'topology_filename': '../FP12-TLR4-MDago_monomer/SYSTEMns.prmtop',
  'coordinates_filename': '../FP12-TLR4-MDago_monomer/PRODUCTION_0/TRAJ-last75ns.nc'},
 {'label': [1],
  'topology_filename': '../FP12-TLR4-MDago_monomer/SYSTEMns.prmtop',
  'coordinates_filename': '../FP12-TLR4-MDago_monomer/PRODUCTION_1/TRAJ-last75ns.nc'},
 {'label': [1],
  'topology_filename': '../FP12-TLR4-MDago_monomer/SYSTEMns.prmtop',
  'coordinates_filename': '../FP12-TLR4-MDago_monomer/PRODUCTION_2/TRAJ-last75ns.nc'}]

In [57]:
import torch
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader

from torch.nn import Embedding, CrossEntropyLoss, Linear, ReLU
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, GATConv
from torch_geometric.nn import radius_graph
from torch_geometric.utils import scatter
from sklearn.model_selection import train_test_split

import MDAnalysis as mda
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import tqdm
import json

## Data Load

In [58]:
# u = mda.Universe("../data/md-2025-03-11/FP11ns.prmtop", "../data/md-2025-03-11/FP11-150ns.nc")
# residue_index = dict((key, id) for id, key in enumerate(np.unique(u.select_atoms('name CA').resnames).tolist()))
# atom_type_index = dict((key, id) for id, key in enumerate(np.unique(u.select_atoms('protein').atoms.types.tolist())))

with open('atom-list.json') as f: 
    atom_type_index = json.load(f)
with open('residue-list.json') as f: 
    residue_index = json.load(f)

In [59]:
def trajectory_to_data(u, selection_string, residue_index, atom_type_index, label):
    for fr in u.trajectory:
        x = torch.from_numpy(u.select_atoms(selection_string).positions)
        residues = list(map(lambda x: residue_index[x], u.select_atoms(selection_string).resnames))
        atom_types = list(map(lambda x: atom_type_index[x], u.select_atoms(selection_string).atoms.types.tolist()))
        d = Data(pos=x, res=torch.tensor(residues, dtype=torch.long), types=torch.tensor(atom_types, dtype=torch.long), label=torch.tensor(label, dtype=torch.long))
        yield d

In [60]:
trj = []

for file in data_files:

    u = mda.Universe(file['topology_filename'], file['coordinates_filename'])
    trj += list(trajectory_to_data(u, selection_string, residue_index, atom_type_index, file['label']))
    print(file['coordinates_filename']) # to check all files are read correctly
    print(len(u.trajectory)) # to check there are 1500 frames per file
    
trj[:5]

../FP11_monomer/PRODUCTION_0/TRAJ-last75ns.nc
750
../FP11_monomer/PRODUCTION_1/TRAJ-last75ns.nc
750
../FP11_monomer/PRODUCTION_2/TRAJ-last75ns.nc
750
../FP12-TLR4-MDago_monomer/PRODUCTION_0/TRAJ-last75ns.nc
750
../FP12-TLR4-MDago_monomer/PRODUCTION_1/TRAJ-last75ns.nc
750
../FP12-TLR4-MDago_monomer/PRODUCTION_2/TRAJ-last75ns.nc
750


[Data(pos=[2244, 3], res=[2244], types=[2244], label=[1]),
 Data(pos=[2244, 3], res=[2244], types=[2244], label=[1]),
 Data(pos=[2244, 3], res=[2244], types=[2244], label=[1]),
 Data(pos=[2244, 3], res=[2244], types=[2244], label=[1]),
 Data(pos=[2244, 3], res=[2244], types=[2244], label=[1])]

## Model

In [61]:
class GCN(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.rg = radius_graph
        self.embed_res = Embedding(num_embeddings=23, embedding_dim=128)
        self.embed_type = Embedding(num_embeddings=33, embedding_dim=128)
        #
        self.conv1_res = GCNConv(128, 64)
        self.relu1_res = ReLU()
        self.conv2_res = GCNConv(64, 32)
        self.relu2_res = ReLU()
        self.conv3_res = GATConv(32, 16)
        # 
        self.conv1_type = GCNConv(128, 64)
        self.relu1_type = ReLU()
        self.conv2_type = GCNConv(64, 32)
        self.relu2_type = ReLU()
        self.conv3_type = GCNConv(32, 16)
        # 
        self.linear1 = Linear(32, 2)

    def forward(self, data):
        pos, res, type, batch = data.pos, data.res, data.types, data.batch
        edge_index = radius_graph(pos, r=5.0, batch=batch)
        #
        z_res = self.embed_res(res)
        z_res = self.conv1_res(z_res, edge_index)
        z_res = self.relu1_res(z_res)
        z_res = self.conv2_res(z_res, edge_index)
        z_res = self.relu2_res(z_res)
        z_res = self.conv3_res(z_res, edge_index)
        #
        z_type = self.embed_type(type)
        z_type = self.conv1_type(z_type, edge_index)
        z_type = self.relu1_type(z_type)
        z_type = self.conv2_type(z_type, edge_index)
        z_type = self.relu2_type(z_type)
        z_type = self.conv3_type(z_type, edge_index)
        #
        z_res = scatter(z_res, data.batch, dim=0, reduce='mean')
        z_type = scatter(z_type, data.batch, dim=0, reduce='mean')
        z = torch.cat([z_res, z_type], dim=1)
        z = self.linear1(z)
        z = F.softmax(z, dim=1)
        return z

In [62]:
train_trj, test_trj = train_test_split(trj, test_size=test_size, random_state=random_state)

In [63]:
print(len(train_trj))

3600


In [64]:
print(len(test_trj))

900


In [65]:
gcn = GCN()
loss = CrossEntropyLoss()
optimizer = torch.optim.Adam(gcn.parameters(), lr=lr, weight_decay=5e-4)
train_loader = DataLoader(train_trj, batch_size=batch_size)
test_loader = DataLoader(test_trj, batch_size=batch_size)
history = []

for i in tqdm.tqdm(range(epochs)):
    statistics = dict()
    statistics['training_loss'] = 0.0
    for d in train_loader:
        z = gcn(d)
        l = loss(z, d.label)
        l.backward()
        optimizer.step()
    statistics['training_loss'] += l.item()
    with torch.no_grad():
        statistics['test_accuracy'] = 0.0
        statistics['test_loss'] = 0.0
        for d in test_loader:
            z = gcn(d)
            l = loss(z, d.label)
            a = ((gcn(d).argmax(1) == d.label).sum())
            statistics['test_accuracy'] += a.item()
            statistics['test_loss'] += l.item()
    statistics['test_accuracy'] /= len(test_trj)
    statistics['test_loss'] /= ((len(test_trj) // batch_size) + 1)
    
    history.append(statistics)

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 15/15 [57:10<00:00, 228.68s/it]


In [66]:
pd.DataFrame.from_records(history)

,training_loss,test_accuracy,test_loss
0,0.668922,0.504444,0.693878
1,0.677463,0.504444,0.693290
2,0.686212,0.504444,0.692931
3,0.691666,0.883333,0.690652
4,0.710071,0.495556,0.677894
5,0.681075,0.676667,0.627501
6,0.480222,0.577778,0.646426
7,0.638620,0.811111,0.512025
8,0.389760,0.802222,0.502694
9,0.331251,0.844444,0.446716


In [ ]:
CHANGE: name="resid 613:754 and name CA N C", using only late simulations, seems to be overfitting due to less frame
    training_loss	test_accuracy	test_loss
0	0.562374	0.513333	0.701102
1	0.773349	0.486667	0.697791
2	0.666513	0.513333	0.679378
3	0.647404	1.000000	0.628916
4	0.648872	0.880000	0.547223
5	0.952839	0.486667	0.697130
6	0.435232	0.513333	0.619875
7	0.790622	0.520000	0.593141
8	0.418558	0.960000	0.462943
9	0.398797	0.973333	0.369488
10	0.340233	0.996667	0.332881
11	0.358396	0.956667	0.360120
12	0.435258	0.933333	0.424054
13	0.348645	0.940000	0.369302
14	0.332083	0.953333	0.356997

CHANGE: full simulations, good results
	training_loss	test_accuracy	test_loss
0	0.687883	0.504444	0.692985
1	0.692517	0.775556	0.691353
2	0.623324	0.640000	0.645995
3	0.506090	0.715556	0.582715
4	0.450717	0.816667	0.498017
5	0.439360	0.824444	0.476845
6	0.345354	0.784444	0.513226
7	0.418391	0.868889	0.436025
8	0.495380	0.842222	0.459707
9	0.314470	0.851111	0.461644
10	0.332093	0.878889	0.427020
11	0.313264	0.864444	0.444329
12	0.425179	0.868889	0.439064
13	0.313262	0.871111	0.438460
14	0.313262	0.913333	0.400578

CHANGE: selection_string="name CA N C", overfitting again
    training_loss	test_accuracy	test_loss
0	0.735092	0.486667	0.696513
1	0.708488	0.486667	0.693901
2	0.706887	0.486667	0.693386
3	0.692634	0.816667	0.690270
4	0.714058	0.486667	0.685859
5	0.655526	1.000000	0.654628
6	0.459406	0.513333	0.678026
7	0.898791	0.486667	0.685773
8	0.539527	0.516667	0.616767
9	0.522720	1.000000	0.520270
10	0.448007	1.000000	0.422417
11	0.491533	0.956667	0.414164
12	0.380476	0.933333	0.413461
13	0.475997	0.923333	0.400632
14	0.339070	0.980000	0.331706

CHANGE: full simulations, again good results overall
	training_loss	test_accuracy	test_loss
0	0.668922	0.504444	0.693878
1	0.677463	0.504444	0.693290
2	0.686212	0.504444	0.692931
3	0.691666	0.883333	0.690652
4	0.710071	0.495556	0.677894
5	0.681075	0.676667	0.627501
6	0.480222	0.577778	0.646426
7	0.638620	0.811111	0.512025
8	0.389760	0.802222	0.502694
9	0.331251	0.844444	0.446716
10	0.361563	0.780000	0.526556
11	0.509104	0.950000	0.364194
12	0.725616	0.856667	0.457344
13	0.359480	0.953333	0.356307
14	0.313727	0.883333	0.424348
